# TAURUS anti-TNF cohort: CD subset, QC and normalisation

scRNA-seq of the TAURUS CD anti-TNF cohort (Thomas et al., 2024).

1. CD biopsies only; QC filters; normalise and log-transform; writes `xnorm_counts_for_aucell.h5ad.gz`
2. `02_taurus_aucell_annotation.R` runs AUCell on that file
3. attaches the AUCell labels and writes `raw_norm_counts_anno.h5ad.gz`, the input of 03-05

Run the cells up to the `xnorm_counts_for_aucell.h5ad.gz` export, then `02_taurus_aucell_annotation.R`, then the rest.

In [ ]:
import scanpy as sc
import pandas as pd
# ── 0. Settings ────────────────────────────────────────────────────────────────
sc.settings.verbosity = 2
sc.settings.n_jobs = 8


In [ ]:
adata = sc.read_h5ad('/path/to/taurus/TAURUS_raw_counts_annotated_final.h5ad')

In [ ]:
cd = adata[adata.obs['Disease']=='CD']

In [ ]:
cd.layers["counts"] = cd.X.copy()

In [ ]:
cd.var["mt"] = cd.var_names.str.startswith("MT-")

In [ ]:
sc.pp.calculate_qc_metrics(
    cd,
    qc_vars=["mt",],
    percent_top=None,
    log1p=False,
    inplace=True,
)
 

In [ ]:
MIN_GENES   = 200     # remove near-empty droplets
MAX_GENES   = 7000    # remove likely doublets (raise for highly complex cells)
MAX_MT_PCT  = 30      # remove dying/lysed cells
MIN_COUNTS  = 500
MAX_COUNTS  = 40000   # dataset-dependent; check the tail
 


In [ ]:
print(f"Cells before filtering: {cd.n_obs}")
 
sc.pp.filter_cells(cd, min_genes=MIN_GENES)
sc.pp.filter_cells(cd, max_genes=MAX_GENES)
sc.pp.filter_cells(cd, min_counts=MIN_COUNTS)
sc.pp.filter_cells(cd, max_counts=MAX_COUNTS)
 
cd = cd[cd.obs["pct_counts_mt"] < MAX_MT_PCT].copy()
 
print(f"Cells after filtering:  {cd.n_obs}")

In [ ]:
sc.pp.filter_genes(cd, min_cells=3)

In [ ]:
sc.pp.normalize_total(cd, target_sum=1e4)
sc.pp.log1p(cd)
 

In [ ]:
cd.write_h5ad("/path/to/taurus/xnorm_counts_for_aucell.h5ad.gz", compression="gzip")

In [ ]:
cd=sc.read_h5ad("/path/to/taurus/xnorm_counts_for_aucell.h5ad.gz")

In [ ]:
anno=pd.read_csv('/path/to/taurus/anno/norm_aucell.csv')

In [ ]:
mapper= pd.read_csv('/path/to/scrna/cd/cell_type_category_map.csv')

In [ ]:
# 1. Clean the names
mapper['cell_type_aucell'] = mapper['cell type'].str.replace(r'[+\-\/()]', ' ', regex=True)
mapper['cell_type_aucell'] = mapper['cell_type_aucell'].str.replace(r'\s+', ' ', regex=True).str.strip()

# 2. Disambiguate duplicates by appending " 1", " 2", etc.
mapper['cell_type_aucell'] = (
    mapper.groupby('cell_type_aucell').cumcount().astype(str).replace('0', '', regex=False)
    .radd(mapper['cell_type_aucell'] + ' ').str.strip()
)

In [ ]:
anno['label_clean'] = anno['label'].str.rstrip()

In [ ]:
anno_map=anno.merge(mapper, how = 'left', left_on = 'label_clean',right_on = 'cell_type_aucell')

In [ ]:
cd.obs['aucell_cell_type_short'] = anno_map['cell type short'].tolist()
cd.obs['aucell_cell_type_full'] = anno_map['cell type'].tolist()
cd.obs['aucell_cell_category_from_type'] = anno_map['cell category'].tolist()
cd.obs['aucell_cell_type_general'] = anno_map['cell type general'].tolist()

In [ ]:
cd.write_h5ad("/path/to/taurus/raw_norm_counts_anno.h5ad.gz", compression="gzip")